In [ ]:
'''
translated_problems-2 부터 translated_problems-10.csv까지의 내용을 취합하는 코드를 작성하고 싶어. 규칙은 다음과 같아. 코드 작성 전에 내 질문에 이해가 안 되는 부분은 질문해줘
- 취합되어 저장할 파일의 이름은 concat_translated_problems.csv로 한다
- 각 파일은 완전 동일한 컬럼과 행(id값)들로 구성되어 있다. 취합할 대상은 generated_title, generated_question, generated_exp_input, generated_exp_output, generated_exp_example 열이다.
- 어차피 행번호(id) 자체는 똑같으므로 ~-2.csv 파일을 기준으로 한 행씩 훑는다. 이때, 각 행 중에는 csv 형식이 깨져서 이상해진 행들이 불규칙하게 존재한다. 그러한 행은 뛰어넘고(아예 생략) 처리하려고 한다. 그러한 행을 판별하는 기준은 raw_tags, tags 열의 값이 리스트인지 판별하는 것이다. 예를 들어 정상적인 행은 해당 열값이 ['dp'], ['Dynamic Programming', '..']처럼 문자열 리스트로 구성되어 있으나 비정상적인 행은 둘중 하나라도 이러한 구조가 아니므로, 그런 행은 뛰어넘는다. -> 이런 행이 총 몇 개인지 이후에 모아서 보고
- 비정상적인 행이 아니라는 가정 하에, 5개 열값이 모두 빈 값이 아니라면 concat에다가 해당 행 자체를 복사해서 넣고 다음 행으로 넘어간다. 5개 열 값은 '모두 비어있거나 모두 비어있지 않아야'한다. 만약 그 중 일부 열만 비어있는 경우가 있다면 그건 비정상적인 행이므로 스킵한다. -> 이러한 행이 총 몇 개이며, 각각 id값과 상위 몇 번째 행인지(1번~ 1씩 증가) 이후에 모아서 보고
- 5개 열값이 모두 빈값인 행이라면 ~-3.csv 파일에서 동일한 행이 모두 가득차있는지 보고, 모두 가득차있다면 마찬가지로 그걸 concat에 복사해넣고 다음 행으로 넘어간다(다시 ~-2.csv로 돌아감). -3도 비어있으면 -4로 넘어가고, 계속 그런식으로 해서 -10까지 보는 방식이다. -10까지 모두 비어있으면 복사 대상이 없는 것이므로 다음 행으로 넘어간다(역시나 ~-2.csv로 돌아감) -> -10까지 모두 비어있는 경우만 모아서 몇 개인지 보고하고, 각각 id값과 상위 몇 번째 행인지 이후에 모아서 보고
- 최종적으로 몇 개의 행들이 concat에 추가되었는지 보고한다.

In [1]:
import pandas as pd
import ast
import os

# ==========================================
# 1. 설정값 및 변수 초기화
# ==========================================
OUTPUT_FILE = 'concat_translated_problems.csv'
TARGET_COLS = ['generated_title', 'generated_question', 'generated_exp_input', 'generated_exp_output', 'generated_exp_example']

# 파일 로드 (translated_problems-2.csv ~ 10.csv)
dfs = []
print("🚀 파일 로딩 중...")
for i in range(2, 11):
    file_name = f'translated_problems-{i}.csv'
    if os.path.exists(file_name):
        dfs.append(pd.read_csv(file_name))
    else:
        print(f"⚠️ 경고: {file_name} 파일이 존재하지 않습니다.")

if not dfs:
    print("❌ 취합할 파일이 없습니다. 프로그램을 종료합니다.")
    exit()

# 보고용 통계 변수
broken_tags_count = 0
partial_empty_info = []  # [(row_num, prob_id), ...]
all_empty_info = []      # [(row_num, prob_id), ...]
concat_rows = []

# ==========================================
# 2. 유틸리티 함수
# ==========================================
def is_valid_list(val):
    """문자열이 정상적인 파이썬 리스트 형태로 파싱되는지 확인"""
    if pd.isna(val): 
        return False
    try:
        parsed = ast.literal_eval(str(val))
        return isinstance(parsed, list)
    except:
        return False

def check_fill_status(row):
    """5개 타겟 컬럼의 상태를 확인하여 (모두채워짐, 모두비어있음, 일부만비어있음) 반환"""
    empty_count = 0
    for col in TARGET_COLS:
        val = row[col]
        # NaN이거나 빈 문자열인 경우
        if pd.isna(val) or str(val).strip() == "":
            empty_count += 1
            
    if empty_count == 0:
        return True, False, False     # all_filled
    elif empty_count == len(TARGET_COLS):
        return False, True, False     # all_empty
    else:
        return False, False, True     # partial_empty

# ==========================================
# 3. 메인 파이프라인 (행 단위 스캔)
# ==========================================
print("🚀 데이터 병합 및 무결성 검사 시작...")
total_rows = len(dfs[0])

for idx in range(total_rows):
    row_num = idx + 1 # 테이블 기준 상위 몇 번째 (1부터 시작)
    base_row = dfs[0].iloc[idx] # 기준 파일 (-2.csv)
    prob_id = base_row['id']
    
    # [조건 1] CSV 형식이 깨진 비정상 행 판별 (raw_tags, tags 검사)
    if not is_valid_list(base_row['raw_tags']) or not is_valid_list(base_row['tags']):
        broken_tags_count += 1
        continue
        
    # [조건 2] 5개 열의 채워짐 상태 확인
    all_filled, all_empty, is_partial = check_fill_status(base_row)
    
    # 일부만 비어있는 비정상 행인 경우
    if is_partial:
        partial_empty_info.append((row_num, prob_id))
        continue
        
    # 모두 가득 차 있는 정상 행인 경우
    if all_filled:
        concat_rows.append(base_row)
        continue
        
    # [조건 3] 모두 비어있는 경우 -> 백업 파일(-3 ~ -10) 순차 탐색
    if all_empty:
        found_filled = False
        
        # dfs[1]부터 끝까지 (-3.csv ~ -10.csv) 
        for file_idx in range(1, len(dfs)):
            fallback_row = dfs[file_idx].iloc[idx]
            f_all_filled, f_all_empty, f_is_partial = check_fill_status(fallback_row)
            
            # 백업 파일에서 완벽하게 꽉 찬 행을 찾았다면
            if f_all_filled:
                concat_rows.append(fallback_row)
                found_filled = True
                break
                
        # -10.csv까지 뒤졌는데도 가득 찬 행을 못 찾은 경우
        if not found_filled:
            all_empty_info.append((row_num, prob_id))

# ==========================================
# 4. 결과 저장 및 통계 보고
# ==========================================
if concat_rows:
    df_concat = pd.DataFrame(concat_rows)
    df_concat.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\n💾 취합 파일 저장 완료: {OUTPUT_FILE}")
else:
    print("\n⚠️ 취합할 수 있는 정상적인 데이터가 단 한 건도 없습니다.")

# 보고서 출력
print("\n" + "="*50)
print("📊 데이터 병합 결과 보고서")
print("="*50)
print(f"✅ 최종적으로 concat에 추가된 행 수: {len(concat_rows)}개")
print(f"❌ 형식 깨짐(tags 파싱 불가)으로 아예 스킵된 행 수: {broken_tags_count}개")
print(f"⚠️ 일부 열만 비어있어(비정상) 스킵된 행 수: {len(partial_empty_info)}개")
print(f"🕳️ -10.csv까지 모든 파일에서 비어있어 추가 실패한 행 수: {len(all_empty_info)}개")

if len(partial_empty_info) > 0:
    print("\n[상세 내역 1] 일부 열만 비어있는 비정상 행 목록 (상위 10개만 출력):")
    for r_num, p_id in partial_empty_info[:10]:
        print(f"  - 테이블 {r_num}번째 행 (id: {p_id})")
    if len(partial_empty_info) > 10: print("  ... 외 생략")

if len(all_empty_info) > 0:
    print("\n[상세 내역 2] -10.csv까지 백업을 찾지 못한 빈 행 목록 (상위 10개만 출력):")
    for r_num, p_id in all_empty_info[:10]:
        print(f"  - 테이블 {r_num}번째 행 (id: {p_id})")
    if len(all_empty_info) > 10: print("  ... 외 생략")
print("="*50)

🚀 파일 로딩 중...


/tmp/ipykernel_55059/3415027215.py:17: DtypeWarning: Columns (0: id, 1: generated_title, 2: generated_question, 3: generated_exp_input, 4: generated_exp_output, 5: generated_exp_example, 6: count_cases, 7: count_solutions, 8: Expected Auxiliary Space, 9: starter_code, 10: picture_num, 11: Unnamed: 21, 12: Unnamed: 22, 13: Unnamed: 23, 14: Unnamed: 25, 15: Unnamed: 26, 16: Unnamed: 30, 17: Unnamed: 32, 18: Unnamed: 36, 19: Unnamed: 50, 20: Unnamed: 54, 21: Unnamed: 56, 22: Unnamed: 60, 23: Unnamed: 80, 24: Unnamed: 84, 25: Unnamed: 92, 26: Unnamed: 102, 27: Unnamed: 110, 28: Unnamed: 114, 29: Unnamed: 120, 30: Unnamed: 126, 31: Unnamed: 140, 32: Unnamed: 144, 33: Unnamed: 150, 34: Unnamed: 156, 35: Unnamed: 164, 36: Unnamed: 170, 37: Unnamed: 173) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(file_name))
/tmp/ipykernel_55059/3415027215.py:17: DtypeWarning: Columns (0: id, 1: generated_title, 2: generated_question, 3: generated_exp_inp

🚀 데이터 병합 및 무결성 검사 시작...

💾 취합 파일 저장 완료: concat_translated_problems.csv

📊 데이터 병합 결과 보고서
✅ 최종적으로 concat에 추가된 행 수: 3542개
❌ 형식 깨짐(tags 파싱 불가)으로 아예 스킵된 행 수: 340개
⚠️ 일부 열만 비어있어(비정상) 스킵된 행 수: 0개
🕳️ -10.csv까지 모든 파일에서 비어있어 추가 실패한 행 수: 752개

[상세 내역 2] -10.csv까지 백업을 찾지 못한 빈 행 목록 (상위 10개만 출력):
  - 테이블 106번째 행 (id: 650)
  - 테이블 340번째 행 (id: 1990)
  - 테이블 343번째 행 (id: 2008)
  - 테이블 368번째 행 (id: 2160)
  - 테이블 376번째 행 (id: 2225)
  - 테이블 381번째 행 (id: 2241)
  - 테이블 382번째 행 (id: 2249)
  - 테이블 391번째 행 (id: 2298)
  - 테이블 430번째 행 (id: 2507)
  - 테이블 436번째 행 (id: 2539)
  ... 외 생략
